In [9]:
# IMPORT LIBRARIES

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [10]:
# LOAD DATASET

df = pd.read_csv("/content/Final_MasterDataset.csv")
print("Dataset loaded successfully!")
print(df.head())
print("\nColumns:", df.columns.tolist())

Dataset loaded successfully!
  Country  Year  D_Expenditure_GDP  Conflict_Intensity  \
0     USA  1994           4.215265                 NaN   
1     USA  1995           3.860246                 NaN   
2     USA  1996           3.554982                 NaN   
3     USA  1997           3.554982                 NaN   
4     USA  1998           3.201558                 NaN   

   Health_Expenditure_(% of GDP)  Education_Expenditure_(% of GDP)  \
0                            NaN                               NaN   
1                            NaN                               NaN   
2                            NaN                               NaN   
3                            NaN                               NaN   
4                            NaN                               NaN   

   Environmental_impact(CO2e/capita)  \
0                              19.83   
1                              19.79   
2                              20.14   
3                              20.90   
4

In [11]:
# SELECT TARGET and FEATURES

target_col = "Education_Expenditure_(% of GDP)"  # TARGET
feature_cols = ["D_Expenditure_GDP", "Year", "Country"]  # FEATURES

In [12]:
# Select only needed columns

data = df[feature_cols + [target_col]].copy()
print("\nBefore dropping missing values:", len(data))


Before dropping missing values: 155


In [13]:
# HANDLE MISSING VALUES

data = data.dropna(subset=[target_col, "D_Expenditure_GDP", "Year", "Country"])
print("After dropping missing values:", len(data))


After dropping missing values: 125


In [14]:
# Split features + target

X = data[feature_cols]
y = data[target_col]

In [15]:
# PREPROCESSING + MODEL PIPELINE

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Country"]),
        ("num", "passthrough", ["D_Expenditure_GDP", "Year"]),
    ]
)


model = Pipeline(steps=[
    ("prep", preprocess),
    ("tree", DecisionTreeRegressor(max_depth=5, random_state=42))
])

In [16]:
# TRAIN,TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 100
Testing rows: 25


In [17]:
# TRAIN MODEL

model.fit(X_train, y_train)
pred = model.predict(X_test)

In [18]:
# EVALUATE MODEL

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5   # SAFE RMSE

In [19]:
print("\n-------- MODEL PERFORMANCE --------")
print("R² Score:", round(r2, 4))
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("-----------------------------------")


-------- MODEL PERFORMANCE --------
R² Score: 0.9465
MAE: 0.2047
RMSE: 0.2702
-----------------------------------


In [20]:
# SHOW EXACT TEST SET ROWS USED FOR TESTING

test_data = X_test.copy()
test_data["Actual_Education_Expenditure"] = y_test.values
test_data["Predicted_Education_Expenditure"] = pred

print("\n-------- TEST SET ROWS (Country, Year, Defense %GDP, Actual, Predicted) --------")
print(test_data)


-------- TEST SET ROWS (Country, Year, Defense %GDP, Actual, Predicted) --------
     D_Expenditure_GDP  Year Country  Actual_Education_Expenditure  \
24            3.304001  2018     USA                      4.895020   
57            4.171479  2020     RUS                      4.016419   
51            4.112993  2014     RUS                      4.013880   
100           2.411122  2001     GBR                      4.100140   
75            1.750165  2007     CHN                      2.700735   
115           1.984491  2016     GBR                      5.427960   
78            1.733520  2010     CHN                      3.559665   
151           1.914406  2021     FRA                      5.417430   
117           1.944556  2018     GBR                      5.204370   
10            4.016313  2004     USA                      6.253762   
41            3.300354  2004     RUS                      3.547870   
144           1.862961  2014     FRA                      5.489204   
59      